# RAG profile matching

This notebook builds a section-level resume index and compares semantic search with hybrid search.

Ground truth for each job is the set of resumes that meet the must-have skills, the minimum years, and the role. Those labels live in `eval_labels.json` and were written with the dataset, before retrieval.

- **Precision@10** is the share of returned resumes that are in the labeled set. If the filter returns fewer than 10, the denominator is the number returned.
- **Recall@10** is the share of labeled resumes that appear in the results.
- **Must-have violations** are returned resumes missing a required skill or below the minimum years.
- **Latency** is the median milliseconds to embed one job and rank resumes, over 3 runs. Index build time is printed separately.


In [1]:
from resume_rag import build_index

stats = build_index()
print(
    "Indexed {resumes} resumes into {chunks} chunks in {seconds} seconds.".format(**stats)
)


Indexed 43 resumes into 215 chunks in 21.5 seconds.


## Section chunks

Nagaraj's resume is stored as separate header, summary, skills, experience, and education chunks. The years line is what the 5+ year filter reads.


In [2]:
from resume_rag import iter_resumes

grouped = iter_resumes()
sample_path = next(path for path in sorted(grouped) if "nagaraj" in path)
sample = grouped[sample_path]
print(sample_path)
print("years:", sample["meta"]["experience_years"])
print("skills:", sample["meta"]["skills"])
print("education:", sample["meta"]["education"])
print()
for chunk in sample["chunks"]:
    print("---", chunk["section"])
    print(chunk["text"])
    print()


resumes/resume_nagaraj.txt
years: 4
skills: Python, Django, Flask, PostgreSQL, Docker
education: B.Tech Computer Science, VTU, 2021

--- header
Nagaraj
Software Engineer | nagaraj@email.com

--- summary
Summary
Software Engineer with 4 years of experience in Python and Django. Built Django REST APIs used by 50000 customers.

--- skills
Skills
Python, Django, Flask, PostgreSQL, Docker

--- experience
Experience
Acme Corp - Software Engineer (2022-present)
Built Django REST APIs used by 50000 customers.
Daily work uses Python, Django, Flask, PostgreSQL, Docker.

--- education
Education
B.Tech Computer Science, VTU, 2021



## Semantic search versus hybrid search

Semantic mode ranks by embedding similarity only. Hybrid mode also requires every must-have skill and the minimum years, then scores 60% similarity, 25% skill overlap, and 15% years.


In [3]:
import json
import time
from pathlib import Path

from job_matcher import match_job_file, parse_job_file, violates_must_have

labels = json.loads(Path("eval_labels.json").read_text(encoding="utf-8"))
catalog = {path: item["meta"] for path, item in grouped.items()}

def time_match(path, hybrid):
    samples = []
    result = None
    for _ in range(3):
        started = time.perf_counter()
        result = match_job_file(path, hybrid=hybrid)
        samples.append((time.perf_counter() - started) * 1000)
    samples.sort()
    return result, samples[len(samples) // 2]

rows = []
header = "{:<28} {:<10} {:>7} {:>7} {:>11} {:>11} {:>9}".format(
    "job", "mode", "P@10", "R@10", "violations", "latency_ms", "returned"
)
print(header)
print("-" * len(header))
for job_path in sorted(Path("jobs").glob("*.txt")):
    job = parse_job_file(job_path)
    relevant = set(labels[job_path.name])
    for mode, hybrid in (("semantic", False), ("hybrid", True)):
        result, elapsed_ms = time_match(job_path, hybrid)
        top = result["top_matches"]
        hits = [item["resume_path"] for item in top if item["resume_path"] in relevant]
        precision = (len(hits) / len(top)) if top else 0.0
        recall = (len(hits) / len(relevant)) if relevant else 0.0
        violations = sum(1 for item in top if violates_must_have(catalog[item["resume_path"]], job))
        row = {
            "job": job_path.stem,
            "mode": mode,
            "precision": precision,
            "recall": recall,
            "violations": violations,
            "latency_ms": elapsed_ms,
            "returned": len(top),
        }
        rows.append(row)
        print(
            "{:<28} {:<10} {:>7.2f} {:>7.2f} {:>11} {:>11.1f} {:>9}".format(
                row["job"],
                row["mode"],
                row["precision"],
                row["recall"],
                row["violations"],
                row["latency_ms"],
                row["returned"],
            )
        )


job                          mode          P@10    R@10  violations  latency_ms  returned
-----------------------------------------------------------------------------------------
data_engineer                semantic      0.50    1.00           5        27.5        10


data_engineer                hybrid        1.00    1.00           0        27.4         5


devops_engineer              semantic      0.50    1.00           5        30.4        10
devops_engineer              hybrid        1.00    1.00           0        29.8         5


frontend_react               semantic      0.50    1.00           5        29.3        10


frontend_react               hybrid        1.00    1.00           0        28.1         5
ios_engineer                 semantic      0.20    1.00           8        29.5        10


ios_engineer                 hybrid        1.00    1.00           0        28.5         2
ml_engineer                  semantic      0.50    1.00           5        29.4        10
ml_engineer                  hybrid        1.00    1.00           0        29.6         5


senior_python_backend        semantic      0.70    0.88           3        29.9        10
senior_python_backend        hybrid        1.00    1.00           0        29.5         8


## One hybrid result

The senior Python job requires Django and 5+ years. Nagaraj has both skills and only 4 years, so he is filtered out.


In [4]:
sample_result = match_job_file("jobs/senior_python_backend.txt")
names = [item["candidate_name"] for item in sample_result["top_matches"]]
print("returned:", ", ".join(names))
print("Nagaraj included:", "Nagaraj" in names)
print()
print(json.dumps(sample_result, indent=2))


returned: Sanjay Kulkarni, Harish Gupta, Meera Krishnan, Pooja Iyer, Arjun Menon, Imran Sheikh, Anita Dsouza, Lakshmi Narayan
Nagaraj included: False

{
  "job_description": "Title: Senior Python Backend Engineer\nMust-have skills: Python, Django\nMinimum years: 5\n\nWe need a backend engineer to design REST APIs in Python and Django backed by PostgreSQL.\nThe candidate must have 5+ years of Python and production Django experience.\nThe work covers Docker deploys, query tuning, and reliable order or payment services.",
  "top_matches": [
    {
      "candidate_name": "Sanjay Kulkarni",
      "resume_path": "resumes/resume_sanjay_kulkarni.pdf",
      "match_score": 84,
      "matched_skills": [
        "Python",
        "Django",
        "PostgreSQL",
        "Docker"
      ],
      "relevant_excerpts": [
        "Skills Python, Django, Docker, Nginx, PostgreSQL",
        "Summary Backend Engineer with 8 years of experience in Python and Django. Containerized the Django API with Docker 

## What the numbers mean


In [5]:
def average(mode, key):
    chosen = [row[key] for row in rows if row["mode"] == mode]
    return sum(chosen) / float(len(chosen))

semantic_precision = average("semantic", "precision")
hybrid_precision = average("hybrid", "precision")
semantic_recall = average("semantic", "recall")
hybrid_recall = average("hybrid", "recall")
semantic_violations = sum(row["violations"] for row in rows if row["mode"] == "semantic")
hybrid_violations = sum(row["violations"] for row in rows if row["mode"] == "hybrid")
mean_latency = average("hybrid", "latency_ms")

print("Mean precision: semantic {0:.2f}, hybrid {1:.2f}.".format(semantic_precision, hybrid_precision))
print("Mean recall: semantic {0:.2f}, hybrid {1:.2f}.".format(semantic_recall, hybrid_recall))
print("Must-have violations: semantic {0}, hybrid {1}.".format(semantic_violations, hybrid_violations))
print("Mean hybrid query latency: {0:.1f} ms.".format(mean_latency))
print()
print(
    "Hybrid search removes resumes that only look close in embedding space. "
    "Semantic search still ranks those neighbors, including people who miss the years bar or a required skill. "
    "The index build above is the one-time cost. Each query after that is the latency row."
)


Mean precision: semantic 0.48, hybrid 1.00.
Mean recall: semantic 0.98, hybrid 1.00.
Must-have violations: semantic 31, hybrid 0.
Mean hybrid query latency: 28.8 ms.

Hybrid search removes resumes that only look close in embedding space. Semantic search still ranks those neighbors, including people who miss the years bar or a required skill. The index build above is the one-time cost. Each query after that is the latency row.
